# Etapa 2. Dados de produtividade e clima

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/02_dados_ibge_clima.ipynb)

Aqui você conhece os dados com que o trabalho inteiro é feito: a **produtividade municipal do IBGE** (Produção
Agrícola Municipal, PAM) e o **clima** da safrinha. O objetivo é responder: quanto o Oeste produz, como a
produtividade evoluiu e quais anos tiveram quebra.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# No Colab, esta célula:
#   1) instala as bibliotecas que o Colab não tem;
#   2) copia o projeto do GitHub (dados + funções) para a máquina do Colab;
#   3) conecta o seu Google Drive, onde os resultados ficam guardados entre uma sessão e outra.
# Fora do Colab (no computador do orientador), ela só acerta as pastas.
import os
import sys
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules        # True quando o notebook roda no Colab

if NO_COLAB:
    !pip -q install agrometeorologiapy==1.0.1
    if Path("/content/milho_oeste").exists():
        !git -C /content/milho_oeste pull -q     # já copiado nesta sessão: só atualiza
    else:
        !git clone -q https://github.com/fcoliveira-utfpr/milho_oeste.git /content/milho_oeste
    os.chdir("/content/milho_oeste")             # entra na pasta do projeto
    from google.colab import drive
    drive.mount("/content/drive")                # pede autorização na primeira vez
    SAIDA = Path("/content/drive/MyDrive/milho_oeste/resultados")
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")
    SAIDA = Path("resultados")

SAIDA.mkdir(parents=True, exist_ok=True)         # cria a pasta de resultados, se não existir
sys.path.append("src")                           # permite importar as funções do projeto (pasta src/)
print("Pasta do projeto:", Path.cwd())
print("Resultados em   :", SAIDA)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

mun = pd.read_csv("dados/municipios_oeste.csv")
ibge = pd.read_csv("dados/ibge_pam_milho_oeste.csv")
clima = pd.read_parquet("dados/clima_oeste.parquet")

ibge = ibge[ibge["safra"] == 2]          # a tabela tem as duas safras; 2 = segunda safra (safrinha)
print(len(mun), "municípios;", ibge["ano"].min(), "a", ibge["ano"].max())
ibge.head()

## 1. Os municípios

A tabela dos municípios traz as coordenadas da sede, a altitude e o solo das áreas de lavoura: a **capacidade de
água disponível** (`dta_mm_m`, mm de água por metro de solo) e os teores de argila e areia. A CAD é o "tamanho da
caixa d'água" do solo no balanço hídrico.

`describe()` resume cada coluna: média, desvio padrão, mínimo, quartis e máximo.

In [ ]:
mun[["altitude_m", "dta_mm_m", "argila_pct", "areia_pct"]].describe().round(1)

## 2. Série regional e tendência tecnológica

`groupby("ano")` junta as linhas de cada ano; `.mean()` tira a média. A reta ajustada (`np.polyfit` de grau 1)
mostra o **ganho tecnológico**: quanto a produtividade cresce por ano por genética e manejo.

Repare que a média simples trata igual um município de 500 ha e outro de 50.000 ha. Por isso também se calcula a
**média ponderada pela área colhida**, que representa a produtividade "da região".

In [ ]:
anual = ibge.groupby("ano").agg(media=("rendimento_kg_ha", "mean"),
                                p90=("rendimento_kg_ha", lambda x: x.quantile(0.90)),
                                producao_t=("producao_t", "sum"),
                                area_ha=("area_colhida_ha", "sum"))
anual["ponderada"] = anual["producao_t"] * 1000 / anual["area_ha"]     # kg/ha

a, b = np.polyfit(anual.index, anual["ponderada"], 1)                  # reta: rend = a·ano + b
print(f"Ganho tecnológico: {a:.0f} kg/ha por ano")
anual.round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(anual.index, anual["ponderada"], marker="o", label="Média ponderada pela área")
ax.plot(anual.index, anual["p90"], marker="s", label="Percentil 90 dos municípios")
ax.plot(anual.index, a * anual.index + b, "--", color="grey", label="Tendência (média ponderada)")
ax.set_ylabel("Produtividade (kg/ha)")
ax.set_title("Milho segunda safra, Oeste do Paraná (IBGE/PAM)")
ax.legend()
fig.savefig(SAIDA / "fig_serie_produtividade.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Os melhores produtores e os anos de quebra

O **percentil 90** de cada ano é o valor abaixo do qual estão 90% dos municípios. Ele representa quem maneja bem a
lavoura, e a variação dele de um ano para outro é sobretudo **clima**. É a referência da calibração da Etapa 6.

A **anomalia** é quanto o ano ficou acima ou abaixo da tendência, em porcentagem. Anomalias muito negativas são
os anos de quebra.

In [ ]:
anual["anomalia_%"] = 100 * (anual["ponderada"] / (a * anual.index + b) - 1)
anual.sort_values("anomalia_%")[["ponderada", "anomalia_%"]].head(6).round(1)

## 4. O clima da safrinha

Para cada ano, a chuva e a temperatura de **fevereiro a junho** (o ciclo da safrinha) na média dos municípios.
`.dt.month` extrai o mês de uma coluna de datas; `.between(2, 6)` testa se está entre 2 e 6.

In [ ]:
clima["ano"] = clima["data"].dt.year
safra = clima[clima["data"].dt.month.between(2, 6)]
por_mun = safra.groupby(["ano", "cod_ibge"]).agg(chuva=("pr", "sum"), eto=("ET", "sum"),
                                                 tmin=("Tmin", "min"), dias_geada=("Tmin", lambda x: (x <= 2).sum()))
clima_ano = por_mun.groupby("ano").mean()                 # média dos 50 municípios
clima_ano["chuva_menos_eto"] = clima_ano["chuva"] - clima_ano["eto"]
tab = clima_ano.join(anual["anomalia_%"]).loc[2003:]
tab.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(tab["chuva_menos_eto"], tab["anomalia_%"])
for ano, r in tab.iterrows():
    ax.annotate(str(ano), (r["chuva_menos_eto"], r["anomalia_%"]), fontsize=8)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("Chuva − ETo de fevereiro a junho (mm)")
ax.set_ylabel("Anomalia da produtividade (%)")
plt.show()
print(f"Correlação: {tab['chuva_menos_eto'].corr(tab['anomalia_%']):.2f}")

### Exercício

1. Quais foram os três piores anos? Procure no site do DERAL/SEAB (boletins de safra) ou em notícias o que
   aconteceu: seca, geada ou calor?
2. Há anos com balanço hídrico ruim e produtividade boa (ou o contrário)? O que poderia explicar? (pense em
   *quando* a falta de água aconteceu, e na geada)
3. Monte uma tabela com os 10 municípios de maior área colhida média (dica: `groupby("municipio")`,
   `["area_colhida_ha"].mean()`, `.sort_values(ascending=False).head(10)`).

### O que escrever

A subseção **Área de estudo e dados** do Material e Métodos: os 50 municípios, a fonte e o período dos dados de
produtividade e de clima, e uma tabela com as características dos municípios.

In [ ]:
# escreva sua resposta aqui